[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/mooseInferenceNotebook.ipynb)

# **MOOSE Inference (Task 1): Download DICOM from IDC, convert to NIfTI, run moosez segmentation**

This notebook is the GPU-side of the MOOSE twoVM workflow. It:
1. Downloads DICOM series from Imaging Data Commons
2. Converts each series to NIfTI via dcm2niix
3. Runs moosez inference with the requested clinical CT models
4. Tars + lz4-compresses the moosez output tree as `moose_segmentations.tar.lz4`

Post-processing (DICOM-SEG generation) happens in Task 2 on a CPU-only VM.

Please cite:

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nat Methods 18, 203-211 (2021). https://doi.org/10.1038/s41592-020-01008-z

Li X, Morgan PS, Ashburner J, Smith J, Rorden C. (2016) The first step for neuroimaging data analysis: DICOM to NIfTI conversion. J Neurosci Methods. 264:47-56.

## Imports

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import yaml
from idc_index.index import IDCClient

## Parameters

The cell below is tagged `parameters` so papermill can inject values at runtime.
When running interactively, edit the values directly.

In [ ]:
# Papermill injects SeriesInstanceUIDs as a Python list when called with
#   papermill notebook.ipynb out.ipynb -y "SeriesInstanceUIDs: [uid1, uid2]"
SeriesInstanceUIDs = [
    "1.3.6.1.4.1.14519.5.2.1.7009.9004.100143549999116733615345241533"
]

# Comma-separated list of moosez clinical CT model names
# Available: clin_ct_body, clin_ct_body_composition, clin_ct_cardiac,
#            clin_ct_digestive, clin_ct_lungs, clin_ct_muscles,
#            clin_ct_organs, clin_ct_peripheral_bones, clin_ct_ribs,
#            clin_ct_vertebrae
moose_models = "clin_ct_body,clin_ct_body_composition,clin_ct_cardiac,clin_ct_digestive,clin_ct_lungs,clin_ct_muscles,clin_ct_organs,clin_ct_peripheral_bones,clin_ct_ribs,clin_ct_vertebrae"

# 'cuda' for GPU, 'cpu' for CPU-only
accelerator = "cuda"


## Normalize parameters

In [ ]:
import re


def _flatten_uids(raw):
    """Normalize any papermill input shape (str / list / dict) to a clean
    list of SeriesInstanceUIDs.

    Tolerates malformed YAML where UIDs are separated by whitespace rather
    than commas, e.g. `[uid1, uid2 uid3 uid4]` — YAML parses the third item
    as one long string, so we re-split on whitespace+commas and drop
    anything that isn't a UID-shaped token (digits and dots only).
    """
    if isinstance(raw, str):
        try:
            parsed = yaml.safe_load(raw)
        except Exception:
            parsed = raw
        if isinstance(parsed, dict):
            parsed = parsed.get("SeriesInstanceUIDs", list(parsed.values()))
        raw = parsed
    items = list(raw) if isinstance(raw, (list, tuple)) else [raw]

    flat = []
    for item in items:
        if item is None:
            continue
        flat.extend(p for p in re.split(r"[\s,]+", str(item).strip()) if p)

    cleaned = [u for u in flat if re.fullmatch(r"[\d.]+", u)]
    seen, deduped = set(), []
    for u in cleaned:
        if u not in seen:
            seen.add(u)
            deduped.append(u)
    return deduped


series_uids = _flatten_uids(SeriesInstanceUIDs)
if not series_uids:
    raise ValueError(
        f"No valid SeriesInstanceUIDs parsed from input: {SeriesInstanceUIDs!r}"
    )

models = [m.strip() for m in moose_models.split(",") if m.strip()]

print(f"Series to process : {len(series_uids)}")
for u in series_uids:
    print(f"  {u}")
print(f"MOOSE models      : {models}")
print(f"Accelerator       : {accelerator}")

## GPU availability check

If `accelerator=cuda` is requested but no GPU is present, fall back to CPU rather than failing.

In [ ]:
usage_metrics = {"gpu": [], "series": {}}

try:
    import nvidia_smi
    nvidia_smi.nvmlInit()
    for i in range(nvidia_smi.nvmlDeviceGetCount()):
        handle = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
        name = nvidia_smi.nvmlDeviceGetName(handle)
        mem = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        usage_metrics["gpu"].append({
            "index": i,
            "name": name if isinstance(name, str) else name.decode(),
            "total_mb": mem.total // (1024 ** 2),
        })
    print(f"GPU(s) found: {usage_metrics['gpu']}")
except Exception as exc:
    print(f"No GPU detected or nvidia-smi unavailable: {exc}")
    if accelerator == "cuda":
        print("WARNING: accelerator=cuda but no GPU found; falling back to cpu")
        accelerator = "cpu"

## Download DICOM from IDC

In [ ]:
DICOM_DIR = Path("/tmp/dicom")
DICOM_DIR.mkdir(parents=True, exist_ok=True)

client = IDCClient()
download_errors = []

for uid in series_uids:
    dest = DICOM_DIR / uid
    dest.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    try:
        client.download_from_selection(
            downloadDir=str(dest),
            seriesInstanceUID=uid,
        )
        usage_metrics["series"].setdefault(uid, {})["download_s"] = round(time.time() - t0, 1)
        print(f"Downloaded {uid} in {usage_metrics['series'][uid]['download_s']}s")
    except Exception as exc:
        msg = f"{uid}: {traceback.format_exc()}"
        download_errors.append(msg)
        print(f"ERROR downloading {uid}: {exc}")

if download_errors:
    Path("download_error_file.txt").write_text("\n".join(download_errors))

## Convert DICOM -> NIfTI with dcm2niix

In [ ]:
NIFTI_DIR = Path("/tmp/nifti")
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

dcm2niix_errors = []
converted_uids = []
failed_download_uids = {e.split(":")[0] for e in download_errors}

for uid in series_uids:
    if uid in failed_download_uids:
        continue

    dcm_path = DICOM_DIR / uid
    nii_path = NIFTI_DIR / uid
    nii_path.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    result = subprocess.run(
        ["dcm2niix", "-z", "y", "-f", "%i_%s", "-o", str(nii_path), str(dcm_path)],
        capture_output=True,
        text=True,
    )
    elapsed = round(time.time() - t0, 1)

    nii_files = list(nii_path.glob("*.nii.gz"))
    if result.returncode != 0 or not nii_files:
        msg = f"{uid}: return_code={result.returncode}\n{result.stderr}"
        dcm2niix_errors.append(msg)
        print(f"ERROR converting {uid}")
    else:
        usage_metrics["series"].setdefault(uid, {})["dcm2niix_s"] = elapsed
        converted_uids.append(uid)
        print(f"Converted {uid} -> {[f.name for f in nii_files]} ({elapsed}s)")

if dcm2niix_errors:
    Path("dcm2niix_error_file.txt").write_text("\n".join(dcm2niix_errors))

print(f"\nSuccessfully converted {len(converted_uids)}/{len(series_uids)} series")

## Run MOOSE inference

moosez writes to `<MOOSE_OUT_DIR>/<uid>/moosez-<model>-<timestamp>/segmentations/` for each
requested model. The whole tree is compressed into a single archive below.

In [ ]:
from moosez import moose

MOOSE_OUT_DIR = Path("/tmp/moose_output")
MOOSE_OUT_DIR.mkdir(parents=True, exist_ok=True)

moose_errors = []

for uid in converted_uids:
    nii_dir = NIFTI_DIR / uid
    out_path = MOOSE_OUT_DIR / uid
    out_path.mkdir(parents=True, exist_ok=True)

    nii_files = list(nii_dir.glob("*.nii.gz"))
    if not nii_files:
        msg = f"{uid}: no NIfTI files found in {nii_dir}"
        moose_errors.append(msg)
        print(f"ERROR: {msg}")
        continue
    nii_file = nii_files[0]

    series_start = time.time()
    series_model_times = {}

    for model in models:
        t0 = time.time()
        try:
            moose(str(nii_file), [model], str(out_path), accelerator)
            elapsed = round(time.time() - t0, 1)
            series_model_times[model] = elapsed
            seg_files = list(out_path.rglob("*.nii.gz"))
            print(f"  {model}: {elapsed}s ({len(seg_files)} mask(s) total so far)")
        except Exception as exc:
            msg = f"{uid}/{model}: {traceback.format_exc()}"
            moose_errors.append(msg)
            print(f"  ERROR {model}: {exc}")

    total_elapsed = round(time.time() - series_start, 1)
    usage_metrics["series"].setdefault(uid, {})["moose_s"] = total_elapsed
    usage_metrics["series"][uid]["moose_models_s"] = series_model_times
    print(f"MOOSE done for {uid}: total {total_elapsed}s across {len(series_model_times)} model(s)")

if moose_errors:
    Path("moose_errors.txt").write_text("\n".join(moose_errors))

print(f"\nMOOSE completed for {len(converted_uids) - len(moose_errors)}/{len(converted_uids)} series")

## Package outputs

In [ ]:
import glob as _glob


def compress_dir(src_dir: Path, out_file: str) -> None:
    """Tar a directory and compress with lz4."""
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    print(f"Created {out_file} ({size_mb:.1f} MB)")


# Extract organ indices (label integer -> name) from each model's dataset.json
# downloaded by moosez, and bundle as moose_organ_indices.json in the archive
# so the post-process task doesn't need moosez installed.
try:
    from moosez.models import MODEL_METADATA
    from moosez import system as _system

    _folder_to_model = {meta["folder_name"]: name for name, meta in MODEL_METADATA.items()}
    organ_indices = {}
    for _json_path in _glob.glob(f"{_system.MODELS_DIRECTORY_PATH}/**/dataset.json", recursive=True):
        _folder = os.path.relpath(_json_path, _system.MODELS_DIRECTORY_PATH).split(os.sep)[0]
        _model_name = _folder_to_model.get(_folder)
        if _model_name is None:
            continue
        with open(_json_path) as _f:
            _labels = json.load(_f).get("labels", {})
        organ_indices[_model_name] = {
            int(k): v for k, v in _labels.items()
            if k.isdigit() and v.lower() not in ("background", "")
        }
    (MOOSE_OUT_DIR / "moose_organ_indices.json").write_text(json.dumps(organ_indices, indent=2))
    print(f"Bundled organ indices for {len(organ_indices)} model(s): {list(organ_indices)}")
except Exception as _exc:
    print(f"WARNING: could not extract organ indices: {_exc}")


compress_dir(MOOSE_OUT_DIR, "moose_segmentations.tar.lz4")


## Write usage metrics

In [ ]:
metrics_json = json.dumps(usage_metrics, indent=2)
metrics_path = Path("moose_inference_UsageMetrics.json")
metrics_path.write_text(metrics_json)

subprocess.run(
    f"lz4 -f {metrics_path} moose_inference_UsageMetrics.lz4",
    shell=True,
    check=True,
)

print(metrics_json)

## Summary

In [ ]:
print("=" * 60)
print("MOOSE Inference Summary")
print("=" * 60)
print(f"  Total series        : {len(series_uids)}")
print(f"  Download failures   : {len(download_errors)}")
print(f"  dcm2niix failures   : {len(dcm2niix_errors)}")
print(f"  MOOSE failures      : {len(moose_errors)}")
print(f"  Models used         : {models}")
print()
for uid, metrics in usage_metrics.get("series", {}).items():
    model_times = metrics.get("moose_models_s", {})
    if model_times:
        print(f"  {uid}")
        for model, secs in model_times.items():
            print(f"    {model:<35} {secs:>7.1f}s")
        print(f"    {'total':<35} {metrics.get('moose_s', 0):>7.1f}s")
print("=" * 60)

if len(moose_errors) == len(series_uids):
    raise RuntimeError("All series failed MOOSE inference - see moose_errors.txt")